The intention of this code is to familiarize mnyself with simulating more standard gauge theories that appear in computational physics. For instance, a special case of $\text{SU}(N)$: $\text{SU}(2)$. This theort can be represented in the fundamental representation via the use of $2 \times 2$ matrices that are all unitary. Thus, we need to store $2 \times L \times L \times N$ elements in the array (The 2 comes from the fact that gauge fields live on the edges).


The action for such a theory is $$ S_E[W_{\mu \nu}] \equiv \beta \sum_{\mu \nu} \Re ( \text{tr}(\mathbb{1} - W_{\mu \nu}(N))) $$ (as it is for all theories that don't have a matter theory; Fermionic matter is notoriously difficult to simulate so I will avoid that at this point). This implies (for our metropolis step), that $$ \Delta S_E [W_{\mu\nu}] = -\beta \sum_{\mu \nu} \Re ( \text{tr}(W_{\mu \nu}(N))) $$


The wilson loops can be defined (in theory) via $$ U_{\mu\nu}(n) = \exp \left\{ i a^2 (\partial_\mu \underline{A}_\nu(n) - \partial_\nu \underline{A}_\mu(n)) + a^2 [\underline{A}_\mu(n), \underline{A}_\nu(n)] + O(a^3) \right\} \\ = \exp \{ i a^2 \underline{F}_{\mu\nu}(n) + O(a^3) \} $$ but this is inconvenient and counterproductive. Instead, we can use $N \times N$ matrices that we exponentiate and randomly update them via the metropolis step (checking how "probable" the update is). Thus, we write 
$$ U_{\mu} = \begin{pmatrix} 
    W_{00\mu} & W_{01\mu} \\
    W_{10\mu} & W_{11\mu}
    \end{pmatrix}
    $$ 
    
then propose an update $$ \Delta U_{\mu} = \begin{pmatrix} \Delta W_{00 \mu} & \Delta W_{01 \mu} \\ \Delta W_{10 \mu} & \Delta W_{11 \mu} \end{pmatrix}$$. 

Thus, $$  U_{\mu} \rightarrow  U_{\mu} + \Delta U_{\mu} $$ or 

This applies for any generic $N \times N$ matrix valued plaquette. To calculate plaquettes, we can use that $$ $$



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup Parameters
L = 16          # Grid size L x L
beta = 1.0      # Coupling constant
sweeps = 1000   # Number of Monte Carlo sweeps
epsilon = 0.5   # Update step size for proposed angles

# Initialize Hot Start (random angles in [-pi, pi))
theta = np.random.uniform(-np.pi, np.pi, size=(2, L, L, 2))

def compute_plaquettes(theta):
    """Computes all 1x1 plaquette angles on an L x L periodic lattice."""
    theta_x = theta[0]
    theta_y = theta[1]
    
    # Correct axis shifts for periodic 2D grid
    theta_y_shift_x = np.roll(theta_y, shift=-1, axis=0) # (x+1, y)
    theta_x_shift_y = np.roll(theta_x, shift=-1, axis=1) # (x, y+1)
    
    return theta_x + theta_y_shift_x - theta_x_shift_y - theta_y

def sweep_metropolis(theta, beta, epsilon):
    """Performs one full Monte Carlo sweep across every link on the lattice."""
    for mu in range(2):
        for x in range(L):
            for y in range(L):
                # Propose a small shift delta
                old_val = theta[mu, x, y]
                delta = np.random.uniform(-epsilon, epsilon)
                new_val = old_val + delta
                
                # Compute local action change Delta S for affected plaquettes only
                # Link at (mu, x, y) affects exactly 2 plaquettes
                if mu == 0:  # x-link
                    # Plaquette 1 (above) and Plaquette 2 (below)
                    p1_old = theta[0, x, y] + theta[1, (x+1)%L, y] - theta[0, x, (y+1)%L] - theta[1, x, y]
                    p2_old = theta[0, x, (y-1)%L] + theta[1, (x+1)%L, (y-1)%L] - theta[0, x, y] - theta[1, x, (y-1)%L]
                    
                    theta[mu, x, y] = new_val
                    p1_new = theta[0, x, y] + theta[1, (x+1)%L, y] - theta[0, x, (y+1)%L] - theta[1, x, y]
                    p2_new = theta[0, x, (y-1)%L] + theta[1, (x+1)%L, (y-1)%L] - theta[0, x, y] - theta[1, x, (y-1)%L]
                else:       # y-link
                    p1_old = theta[0, x, y] + theta[1, (x+1)%L, y] - theta[0, x, (y+1)%L] - theta[1, x, y]
                    p2_old = theta[0, (x-1)%L, y] + theta[1, x, y] - theta[0, (x-1)%L, (y+1)%L] - theta[1, (x-1)%L, y]
                    
                    theta[mu, x, y] = new_val
                    p1_new = theta[0, x, y] + theta[1, (x+1)%L, y] - theta[0, x, (y+1)%L] - theta[1, x, y]
                    p2_new = theta[0, (x-1)%L, y] + theta[1, x, y] - theta[0, (x-1)%L, (y+1)%L] - theta[1, (x-1)%L, y]
                
                # Delta S = -beta * (cos(new) - cos(old))
                dS = -beta * ((np.cos(p1_new) + np.cos(p2_new)) - (np.cos(p1_old) + np.cos(p2_old)))
                
                # Metropolis Accept / Reject step
                if dS > 0 and np.random.rand() > np.exp(-dS):
                    theta[mu, x, y] = old_val # Reject: revert back

# 2. Main Thermalization Loop
history = []
print("Thermalizing lattice...")
for step in range(sweeps):
    sweep_metropolis(theta, beta, epsilon)
    avg_plaq = np.mean(np.cos(compute_plaquettes(theta)))
    history.append(avg_plaq)

# 3. Plot Thermalization Curve
plt.figure(figsize=(8, 4))
plt.plot(history, color='crimson', lw=1.5)
plt.axhline(0, color='gray', linestyle='--', label='Hot Start Baseline (0.0)')
plt.xlabel("Monte Carlo Sweeps")
plt.ylabel(r"Average Plaquette $\langle \cos(\theta_{\mathrm{plaq}}) \rangle$")
plt.title(f"U(1) Lattice Gauge Thermalization (L={L}, $\\beta={beta}$)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

